In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit

In [14]:
DATASET = "../data/dataset/hinglish-dataset.csv"
PROCESSED_DATASET = "../data/processed/hinglish-dataset.csv"

In [15]:
df = pd.read_csv(DATASET)

In [16]:
# Encoding the labels
le = LabelEncoder()

y = le.fit_transform(df['label'])
labels = le.classes_

print(y)
print(dict(zip(labels, le.transform(labels))))

[3 3 3 ... 3 3 3]
{'Hate Speech': np.int64(0), 'Obviously Threatening': np.int64(1), 'Offensive': np.int64(2), 'Safe': np.int64(3)}


In [17]:
# Stratified Shuffle Split

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(splitter.split(df, y))

In [18]:
# Create training and test datasets using positional indices
X_train = df["text"].iloc[train_idx].reset_index(drop=True)
X_test = df["text"].iloc[test_idx].reset_index(drop=True)
y_train = y[train_idx]
y_test = y[test_idx]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")


Training samples: 16913
Test samples: 4229


In [19]:
# Save encoded labels and split assignments alongside the original data
processed_df = df.copy()
processed_df["label_id"] = y
processed_df["split"] = "train"
processed_df.iloc[test_idx, processed_df.columns.get_loc("split")] = "test"
processed_df.to_csv(PROCESSED_DATASET, index=False)

# Save the mapping for decoding model predictions later
from pathlib import Path
import json

mapping_path = Path(PROCESSED_DATASET).with_name("label_mapping.json")
label_mapping = {label: int(index) for index, label in enumerate(labels)}
mapping_path.write_text(json.dumps(label_mapping, indent=2), encoding="utf-8")
print(f"Saved dataset to {PROCESSED_DATASET}")
print(f"Saved label mapping to {mapping_path}")


Saved dataset to ../data/processed/hinglish-dataset.csv
Saved label mapping to ..\data\processed\label_mapping.json
